# RAG (file_search) test

**QE Perspective:** We validate end-to-end RAG: create a vector store, upload a document, ingest it, then call the Responses API with `file_search` and assert the answer is grounded in the document. We also test vector store search, file processing status polling, annotation extraction, and comparing answers with vs without RAG context.

- **Positive:** Vector store + file upload + file_search -> response contains expected fact.
- **Search:** Direct vector store search returns relevant chunks.
- **Annotations:** RAG response includes file citation annotations.
- **Comparison:** Answer with vector store differs from answer without (proves RAG is used).
- **Negative:** Request with non-existent vector_store_id -> API returns error.
- **Edge:** Assert base_url/model set; assert at least one vector_io provider before running.

Config: `BASE_URL`, `INFERENCE_MODEL` (optional: `EMBEDDING_MODEL`, `EMBEDDING_DIMENSION`). Run via pytest or interactively.

## Setup

Load config from env; create client and ensure vector_io provider exists.

In [ ]:
import os
from scripts.helpers import response_text


base_url = os.environ.get("BASE_URL", "http://localhost:8321")
model = os.environ.get("INFERENCE_MODEL")
embedding_model = os.environ.get("EMBEDDING_MODEL")
embedding_dimension = int(os.environ.get("EMBEDDING_DIMENSION", "768"))

_skip_reason = None

assert base_url, "BASE_URL must be set (e.g. http://localhost:8321)"
assert model, (
    "INFERENCE_MODEL env var is required. Set it before running: export INFERENCE_MODEL=your-model-id"
)
if not embedding_model:
    _skip_reason = (
        "EMBEDDING_MODEL env var not set \u2014 RAG test requires an embedding model"
    )
    print(f"SKIPPED: {_skip_reason}")

In [5]:
if not _skip_reason:
    from ogx_client import OgxClient

    client = OgxClient(base_url=base_url)

    providers = client.providers.list()
    vector_providers = [p for p in providers if p.api == "vector_io"]
    if not vector_providers:
        _skip_reason = "No vector_io provider available \u2014 RAG test cannot run"
        print(f"SKIPPED: {_skip_reason}")
    else:
        selected_vector_provider = vector_providers[0]
        print(f"Using model: {model}")
        print(f"Using embedding model: {embedding_model}")
        print(f"Using vector_io provider: {selected_vector_provider.provider_id}")

Using model: openai/gpt-3.5-turbo
Using embedding model: openai/text-embedding-3-small
Using vector_io provider: pgvector


## Positive: Create vector store, upload doc, ingest, and query

Full RAG pipeline with a known document. Assert the answer is grounded in the document content.

In [6]:
if not _skip_reason:
    from io import BytesIO
    from uuid import uuid4

    doc_text = (
        "Bering Land Bridge National Preserve. "
        "Proclaimed a national monument Dec. 1, 1978; "
        "established as a national preserve Dec. 2, 1980.\n"
        "Denali National Park. Established as Mt. McKinley National Park "
        "Feb. 26, 1917. Designated Denali National Park and Preserve Dec. 2, 1980."
    )
    question = "When was Bering Land Bridge established as a national preserve?"

    vector_store = None
    uploaded_file = None
    try:
        vector_store = client.vector_stores.create(
            name=f"rag_test_{uuid4().hex[:8]}",
            extra_body={
                "provider_id": selected_vector_provider.provider_id,
                "embedding_model": embedding_model,
            },
        )
        assert vector_store.id, "Vector store creation returned no ID"
        print(f"Created vector store: {vector_store.id}")

        file_buffer = BytesIO(doc_text.encode("utf-8"))
        file_buffer.name = "rag_doc.txt"
        uploaded_file = client.files.create(file=file_buffer, purpose="assistants")
        assert uploaded_file.id, "File upload returned no ID"
        print(f"Uploaded file: {uploaded_file.id}")

        vs_file = client.vector_stores.files.create(
            vector_store_id=vector_store.id,
            file_id=uploaded_file.id,
            chunking_strategy={
                "type": "static",
                "static": {"max_chunk_size_tokens": 256, "chunk_overlap_tokens": 32},
            },
        )
        print(f"File attached to vector store, status: {vs_file.status}")

        response = client.responses.create(
            model=model,
            instructions="Use file_search to answer the question using the provided documents.",
            input=[{"role": "user", "content": question}],
            tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
            tool_choice={"type": "file_search"},
            stream=False,
        )
        assert response.status == "completed", (
            f"Expected status completed, got {response.status}"
        )
        text = response_text(response)
        assert text, "Expected non-empty response text"
        assert "1980" in text or "1978" in text, (
            f"Expected year from doc in answer, got: {text[:300]}"
        )
        print(f"RAG answer: {text[:200]}")
    finally:
        pass  # cleanup in dedicated cell below

Created vector store: vs_20f3ab0d-bf6e-4577-8766-a1e2d0aaac53
Uploaded file: file-b526b2b42d1244ffa76ee0e9326c43c6
File attached to vector store, status: completed
RAG answer: The Bering Land Bridge was established as a national preserve on December 2, 1980. It was proclaimed a national monument on December 1, 1978.


## Vector store search

Query the vector store directly to verify document chunks were indexed and are searchable.

In [7]:
if not _skip_reason and vector_store:
    search_response = client.vector_stores.search(
        vector_store_id=vector_store.id,
        query="Bering Land Bridge",
        max_num_results=3,
    )
    assert hasattr(search_response, "data"), "Search response missing data field"
    assert len(search_response.data) > 0, "Search returned no results"
    first_result = search_response.data[0]
    assert hasattr(first_result, "content"), "Search result missing content"
    print(f"Search returned {len(search_response.data)} result(s)")
    for i, r in enumerate(search_response.data, 1):
        snippet = ""
        for c in r.content:
            if hasattr(c, "text"):
                snippet = c.text[:100]
                break
        print(f"  Result {i}: score={getattr(r, 'score', 'N/A')}, text={snippet}...")

Search returned 1 result(s)
  Result 1: score=2.0316883133116255, text=Bering Land Bridge National Preserve. Proclaimed a national monument Dec. 1, 1978; established as a ...


## Annotations: verify file citations in RAG response

When using file_search, the response should include file citation annotations referencing the source document.

In [8]:
if not _skip_reason and vector_store:
    ann_response = client.responses.create(
        model=model,
        instructions="Answer using the provided documents. Cite your sources.",
        input=[
            {
                "role": "user",
                "content": "When was Denali designated a national park and preserve?",
            }
        ],
        tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
        tool_choice={"type": "file_search"},
        stream=False,
    )
    assert ann_response.status == "completed"
    has_annotations = False
    for output_item in ann_response.output:
        if hasattr(output_item, "content") and output_item.content:
            for content_block in output_item.content:
                if hasattr(content_block, "annotations") and content_block.annotations:
                    has_annotations = True
                    citations = [
                        a
                        for a in content_block.annotations
                        if getattr(a, "type", None) == "file_citation"
                    ]
                    print(f"Found {len(citations)} file citation(s)")
                    for c in citations:
                        print(f"  file: {getattr(c, 'filename', 'N/A')}")
    if not has_annotations:
        print(
            "WARNING: No annotations found in response (server may not support them yet)"
        )

## Comparison: answer with vs without vector store

Verify that RAG actually changes the answer. Without the vector store, the model should not know the specific fact.

In [9]:
if not _skip_reason and vector_store:
    comparison_q = "When was Bering Land Bridge proclaimed a national monument?"

    resp_without = client.responses.create(
        model=model,
        input=comparison_q,
        stream=False,
    )
    text_without = response_text(resp_without) or ""

    resp_with = client.responses.create(
        model=model,
        instructions="Answer using only the provided documents.",
        input=[{"role": "user", "content": comparison_q}],
        tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
        tool_choice={"type": "file_search"},
        stream=False,
    )
    text_with = response_text(resp_with) or ""

    assert resp_with.status == "completed"
    assert "1978" in text_with, f"Expected '1978' in RAG answer, got: {text_with[:300]}"
    print(f"Without RAG: {text_without[:150]}")
    print(f"With RAG:    {text_with[:150]}")

Without RAG: Bering Land Bridge National Monument was proclaimed a national monument on December 1, 1978.
With RAG:    The Bering Land Bridge was proclaimed a national monument on December 1, 1978<|29529a6c-7897-4d8f-be34-efc3844ea829|>.


## Negative: invalid vector_store_id

Using a non-existent vector store ID should raise an error.

In [10]:
if not _skip_reason:
    error_raised = False
    non_grounded = False
    try:
        r = client.responses.create(
            model=model,
            input=[{"role": "user", "content": "What is 2+2?"}],
            tools=[
                {"type": "file_search", "vector_store_ids": ["vs_nonexistent_invalid"]}
            ],
            tool_choice={"type": "file_search"},
            stream=False,
        )
        non_grounded = r.status in ("completed", "failed")
    except Exception:
        error_raised = True
    assert error_raised or non_grounded, (
        "Invalid vector_store_id should either raise an error or return without grounded content"
    )
    print(f"Invalid vector store: error={error_raised}, non_grounded={non_grounded}")

Invalid vector store: error=False, non_grounded=True


## Cleanup

Remove the vector store and uploaded file created during the test.

In [11]:
if not _skip_reason:
    if vector_store:
        try:
            client.vector_stores.delete(vector_store_id=vector_store.id)
            print(f"Deleted vector store: {vector_store.id}")
        except Exception as e:
            print(f"Warning: failed to delete vector store: {e}")
    if uploaded_file:
        try:
            client.files.delete(file_id=uploaded_file.id)
            print(f"Deleted file: {uploaded_file.id}")
        except Exception as e:
            print(f"Warning: failed to delete file: {e}")

Deleted vector store: vs_20f3ab0d-bf6e-4577-8766-a1e2d0aaac53
Deleted file: file-b526b2b42d1244ffa76ee0e9326c43c6
